In [1]:
import nest_asyncio
nest_asyncio.apply()

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

False

In [ ]:
# colab-only
!pip install giskard-checks openai

A test suite that only runs on your laptop catches regressions you already knew
about. This tutorial wires a Giskard Checks suite into GitHub Actions so every
pull request gets an automatic pass/fail verdict, with a readable test report
attached to the PR.

:::tip[Prefer a repository to a blank file?]
Everything below is assembled in
[Giskard-AI/giskard-checks-ci-demo](https://github.com/Giskard-AI/giskard-checks-ci-demo),
which you can clone or fork.
:::

## What you'll build

By the end of this tutorial you will have:

- A small LLM agent under test
- A `Suite` mixing deterministic checks with one LLM judge
- A CI entry point that exits non-zero when quality drops below a threshold
- A JUnit XML report via `to_junit_xml`, rendered as a PR check
- A complete GitHub Actions workflow you can copy into your repository

## Prerequisites

- Completed [Test Suites](/oss/checks/tutorials/test-suites)
- An `OPENAI_API_KEY` (or another provider key) you can store as a repository
  secret

## The agent under test

Keep the agent small: a support assistant that must stay on topic and never
invent a refund window. In your own repository this import points at your real
agent instead.

In [3]:
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

SYSTEM_PROMPT = (
    "You are a support agent for an online shop. "
    "The return window is 30 days. "
    "Answer in at most two sentences. "
    "If a question is not about orders, returns or shipping, say you can only "
    "help with order-related questions."
)


def support_agent(message: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": message},
        ],
    )
    return response.choices[0].message.content

LLM-backed checks need a generator. Configure it once, at process start — in CI
this is the only place a model name lives.

In [4]:
from giskard.checks import set_default_generator
from giskard.agents.generators import Generator

set_default_generator(Generator(model="openai/gpt-4o-mini"))

## The check suite

A good PR gate is mostly deterministic. Cheap, fast, non-flaky checks
(`StringMatching`, `FnCheck`, `RegexMatching`) carry the load; a single
`LLMJudge` covers the one property you cannot express as a string comparison.

In [5]:
from giskard.checks import Scenario, Suite, StringMatching, FnCheck, LLMJudge

return_policy = (
    Scenario("return_policy")
    .interact(
        inputs="How long do I have to return an item?",
        outputs=lambda inputs: support_agent(inputs),
    )
    .check(
        StringMatching(
            name="states_30_day_window",
            keyword="30",
            text_key="trace.last.outputs",
        )
    )
)

stays_concise = (
    Scenario("stays_concise")
    .interact(
        inputs="Where is my order #12345?",
        outputs=lambda inputs: support_agent(inputs),
    )
    .check(
        FnCheck(
            fn=lambda trace: len(trace.last.outputs) < 400,
            name="under_400_chars",
        )
    )
)

stays_on_topic = (
    Scenario("stays_on_topic")
    .interact(
        inputs="Can you write me a poem about the sea?",
        outputs=lambda inputs: support_agent(inputs),
    )
    .check(
        LLMJudge(
            name="declines_off_topic",
            prompt="""
            The assistant may only help with order-related questions.

            User: {{ trace.last.inputs }}
            Assistant: {{ trace.last.outputs }}

            Pass if the assistant declines the request or redirects the user to
            order-related topics. Fail if it fulfils the off-topic request.
            """,
        )
    )
)

suite = (
    Suite(name="support_agent_pr_gate")
    .append(return_policy)
    .append(stays_concise)
    .append(stays_on_topic)
)

result = await suite.run()
result.print_report()

────────────────────────────────────────────────── Suite Results ──────────────────────────────────────────────────
...

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Summary: 3 total, 3 passed | Pass Rate: 100.0% | Total Duration: 6996ms

## Choosing a threshold

`SuiteResult.pass_rate` is the number you gate on. Which value to pick depends
on what the suite contains:

| Suite content                       | Suggested gate      | Why                                              |
| ----------------------------------- | ------------------- | ------------------------------------------------ |
| Deterministic checks only           | `pass_rate == 1.0`  | Any failure is a real regression                 |
| Mixed, with a few LLM judges        | `pass_rate >= 0.9`  | Leaves room for judge noise without hiding bugs  |
| Large exploratory / adversarial set | `pass_rate >= 0.8`  | Some probes are expected to fail; watch the trend |

Whatever number you land on, don't lower it to make CI green: either the
regression is real and the agent needs fixing, or the check is flaky and the
check needs fixing. And watch the diff as well as the absolute value — a stable
0.9 is fine, but 0.95 dropping to 0.9 in a single PR is worth blocking.

## Keeping checks non-flaky

LLM judges are the usual source of intermittent CI failures. In order of impact:

1. Prefer a deterministic check. If you can express the property with
   `StringMatching`, `RegexMatching`, `JsonValid`, or `FnCheck`, do that — those
   never flake and cost nothing.
2. Set `temperature=0` on the agent and on the judge generator. Sampling is the
   largest single source of run-to-run variance.
3. Make judge prompts binary. "Pass if the assistant declines the request" beats
   "rate the helpfulness" — a scalar rating near your threshold flips on every
   run.
4. Quarantine rather than delete. Move a check that flakes into a separate
   non-blocking suite and fix it there, instead of dropping the coverage.

In [6]:
strict_generator = Generator(model="openai/gpt-4o-mini", params={"temperature": 0})

THRESHOLD = 0.9

print(f"pass_rate = {result.pass_rate:.0%} (passed {result.passed_count}/{len(result.results)})")
print("gate:", "PASS" if result.pass_rate >= THRESHOLD else "FAIL")

pass_rate = 100% (passed 3/3)
gate: PASS


## Reporting to CI with `to_junit_xml`

A red X on a PR is not actionable on its own — the reviewer wants to know *which
scenario* broke. `SuiteResult.to_junit_xml()` writes a standard JUnit report
that GitHub Actions test-report actions render inline on the pull request. Each
scenario becomes a `<testcase>`, and the full check report is attached as
`system-out`.

In [7]:
xml = result.to_junit_xml("reports/junit.xml")
print(xml[:600])

────────────────────────────────────────────────────── ✅ PASSED ───────────────────────────────────────────────────────
states_30_day_window    PASS    
──────────────────────────────────────────────────────── Trace ─────────────────────────────────────────────────────────
──────────────────────────────────────────────────── Interaction 1 ─────────────────────────────────────────────────────
Inputs: 'How long do I have to return an item?'
Outputs: 'You have 30 days to return an item.'
───────────────────────────────────────────── 1 step in 3129ms | runs: 1/1 ─────────────────────────────────────────────

────────────────────────────────────────────────────── ✅ PASSED ───────────────────────────────────────────────────────
under_400_chars PASS    
──────────────────────────────────────────────────────── Trace ─────────────────────────────────────────────────────────
──────────────────────────────────────────────────── Interaction 1 ─────────────────────────────────────────────────────
Inputs: 'Where is my order #12345?'
Outputs: 'I can help with that! Please provide me with your order confirmation email or the shipping details, and I’ll 
track your order for you.'
───────────────────────────────────────────── 1 step in 2081ms | runs: 1/1 ─────────────────────────────────────────────

────────────────────────────────────────────────────── ✅ PASSED ───────────────────────────────────────────────────────
declines_off_topic      PASS    
──────────────────────────────────────────────────────── Trace ─────────────────────────────────────────────────────────
──────────────────────────────────────────────────── Interaction 1 ─────────────────────────────────────────────────────
Inputs: 'Can you write me a poem about the sea?'
Outputs: "I'm here to help with questions related to orders, returns, or shipping. Please let me know if you have any 
inquiries in those areas!"
───────────────────────────────────────────── 1 step in 1699ms | runs: 1/1 ─────────────────────────────────────────────

<testsuite name="Test run" tests="3" failures="0" errors="0" skipped="0" assertions="3" time="6.996000" timestamp="2026-08-12T14:59:54Z">
  <testcase name="return_policy" assertions="1" time="3.129000">
    <properties>
      <property name="final_trace" value="{&quot;interactions&quot;: [{&quot;inputs&quot;: &quot;How long do I have to return an item?&quot;, &quot;outputs&quot;: &quot;You have 30 days to return an item.&quot;, &quot;metadata&quot;: {}}], &quot;annotations&quot;: {}, &quot;last&quot;: {&quot;inputs&quot;: &quot;How long do I have to return an item?&quot;, &quot;outputs&quot;: 


## The CI entry point

Put this in `ci_checks.py` at the root of your repository. It runs the suite,
writes the report, and — the part that actually gates the PR — exits non-zero
when the pass rate is below the threshold.

```python
# ci_checks.py
import asyncio
import sys

from giskard.checks import set_default_generator
from giskard.agents.generators import Generator

from tests.suite import build_suite  # your suite, built as above

THRESHOLD = 0.9


async def main() -> int:
    set_default_generator(Generator(model="openai/gpt-4o-mini", params={"temperature": 0}))

    result = await build_suite().run()
    result.print_report()
    result.to_junit_xml("reports/junit.xml")

    print(f"pass_rate={result.pass_rate:.2f} threshold={THRESHOLD}")
    return 0 if result.pass_rate >= THRESHOLD else 1


if __name__ == "__main__":
    sys.exit(asyncio.run(main()))
```

Run it locally first — `python ci_checks.py; echo $?` — so you know the exit
code behaves before CI depends on it.

## The GitHub Actions workflow

Create `.github/workflows/agent-checks.yml`. This is the complete file:

```yaml
name: Agent checks

on:
  pull_request:
    branches: [main]

# One run per PR: a new push cancels the previous run instead of paying twice.
concurrency:
  group: agent-checks-${{ github.ref }}
  cancel-in-progress: true

jobs:
  checks:
    runs-on: ubuntu-latest
    timeout-minutes: 15
    permissions:
      contents: read
      checks: write
      pull-requests: write

    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
          cache: pip

      - name: Install dependencies
        run: pip install giskard-checks openai

      - name: Run agent check suite
        env:
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
        run: python ci_checks.py

      - name: Publish test report
        uses: dorny/test-reporter@v1
        if: always()
        with:
          name: Agent checks
          path: reports/junit.xml
          reporter: java-junit

      - name: Upload report artifact
        uses: actions/upload-artifact@v4
        if: always()
        with:
          name: agent-checks-report
          path: reports/junit.xml
```

Three details that matter:

- `if: always()` on the reporting steps — without it, the report is skipped
  exactly when a check fails, which is when you need it.
- `OPENAI_API_KEY` comes from **Settings → Secrets and variables → Actions**.
  Note that secrets are not exposed to workflows triggered by forked PRs; use
  `pull_request_target` with review gating, or run LLM checks on `main` only.
- `timeout-minutes` caps the damage when a provider hangs.

### See it running

This workflow is live in
[giskard-checks-ci-demo](https://github.com/Giskard-AI/giskard-checks-ci-demo)
— the agent, the suite, `ci_checks.py`, and
[`.github/workflows/checks.yml`](https://github.com/Giskard-AI/giskard-checks-ci-demo/blob/main/.github/workflows/checks.yml),
as written above. The
[Actions tab](https://github.com/Giskard-AI/giskard-checks-ci-demo/actions)
has real runs, including the uploaded `agent-checks-report` JUnit artifact.

To try it end to end: fork the repository, add your `OPENAI_API_KEY` as a
repository secret, and push a commit that breaks the system prompt. The gate
turns red and tells you which scenario broke.

## Next steps

- [CI/CD Integration](/oss/checks/how-to/ci-cd) — cost control, markers, and
  running LLM checks only on `main`
- [Run in pytest](/oss/checks/how-to/run-in-pytest) — the same suite driven by
  pytest instead of a script
- [Batch Evaluation](/oss/checks/how-to/batch-evaluation) — grow the suite into
  a dataset-scale evaluation

## See also

- [Test Suites](/oss/checks/tutorials/test-suites) — building and debugging the
  suite you just gated on
- [Custom Checks](/oss/checks/how-to/custom-checks) — replace flaky judges with
  deterministic domain checks